# ScreamingFace client tour

Explore the full public Client surface without making a paid model call. This complements the
short quickstart: it covers explicit Client lifecycle, Engine discovery, provider connections,
Model and Fusion authoring, hosted authentication, asynchronous use, progress Events, typed
errors, and Report anatomy.

Every state-changing or paid example is either descriptive or guarded off by default.

## Before running

From a terminal:

```bash
screamingface prepare draco  # first run only: download pinned Benchmark assets
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

## 1. Choose a Client lifecycle

Module functions such as `sf.models.list()` use one lazy default Client. `sf.configure()`
replaces
that default when an application needs another Engine origin, and `sf.close()` closes it.

Long-running applications can instead own an explicit Client and close it deterministically.
This
tour uses that form so its lifecycle is visible.

In [ ]:
client = sf.Client(engine_url="http://127.0.0.1:9108")
{
    "engine_url": client.engine_url,
    "closed": client.closed,
    "authenticated": client.authenticated,
    "authenticating": client.authenticating,
}

For a hosted Engine protected by Cloudflare Access, caller login is separate from
provider credentials. Protected requests can start login automatically, or an application can be
explicit:

```python
with sf.Client(engine_url="https://your-engine.example") as hosted:
    hosted.login(timeout=300)
    print(hosted.authenticated)
    hosted.logout()
```

Local loopback development does not require that browser flow.

## 2. Discover Models and their exact contracts

In [ ]:
models = client.models.list()
models

In [ ]:
MODEL_ID = "openrouter/google/gemini-3-flash-preview"
model = client.models.get(MODEL_ID)
{
    "id": model.id,
    "provider": model.provider,
    "auth_mode": model.auth_mode,
    "enabled_parameters": [
        name for name, parameter in model.parameters.items() if parameter.enabled
    ],
    "enabled_tools": [
        name for name, capability in model.tools.items() if capability.gateway_status == "enabled"
    ],
    "stale": model.stale,
    "degraded": model.degraded,
}

Parameter schemas are executable contracts. Candidate construction is local;
evaluation preflight validates the selected values against this live Engine contract before any
model request is launched.

In [ ]:
max_tokens = model.parameters["max_tokens"]
{
    "request_path": max_tokens.request_path,
    "schema": max_tokens.schema,
    "provider_support": max_tokens.provider_support,
    "gateway_projection": max_tokens.gateway_projection,
    "cache_behavior": max_tokens.cache_behavior,
}

## 3. Discover Benchmarks

In [ ]:
benchmarks = client.benchmarks.list()
benchmarks

In [ ]:
draco = client.benchmarks.get("draco")
{
    "id": draco.id,
    "title": draco.title,
    "description": draco.description,
    "revision": draco.revision,
    "case_count": draco.case_count,
}

### Module-level shorthand

Every discovery call above has a module-level form backed by the one lazy default Client.
Use the explicit Client when you need lifecycle control; use these in a notebook.

In [ ]:
sf.benchmarks.list()
sf.benchmarks.get("draco")
sf.models.get("openrouter/google/gemini-3-flash-preview")

## 4. Inspect and manage provider connections

`client.connect()` displays the Engine-backed notebook panel. Applications can also use
`client.connect("openrouter", api_key=...)`, OAuth, `client.connections.get(...)`, and
`client.disconnect(...)`. Provider secrets go to the Engine for validation and encrypted
storage;
they are never returned by discovery.

In [ ]:
client.connections.list()

In [ ]:
MUTATE_CONNECTIONS = False

if MUTATE_CONNECTIONS:
    from getpass import getpass

    connection = client.connect("openrouter", api_key=getpass("OpenRouter API key: "))
else:
    connection = "Connection mutation disabled. Use client.connect() for the notebook panel."
connection

OAuth providers return a bounded flow rather than a secret:

```python
flow = client.connect("provider-id", method="oauth")
print(flow.authorize_url)
connection = flow.wait(timeout=300)  # or flow.cancel()
client.disconnect("provider-id")
```

## 5. Author Models and Fusions locally

In [ ]:
writer = sf.Model(
    MODEL_ID,
    name="writer",
    prompt="Answer accurately and explain the important trade-offs.",
    params={"max_tokens": 4096, "temperature": 0.0},
)
reviewer = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    name="reviewer",
    params={"max_tokens": 4096, "temperature": 0.0},
)
panel = sf.Fusion(
    [writer, reviewer],
    name="reviewed-answer",
    synthesizer=sf.Model(
        MODEL_ID,
        prompt="Produce one accurate final answer from the panel responses.",
        params={"max_tokens": 4096, "temperature": 0.0},
    ),
)
[writer, panel]

Recipes contain no Benchmark logic. At evaluation time the Client compiles each
Recipe into URL4 and links it to the selected Engine-owned Benchmark protocol.

## 6. Evaluate with progress and typed Events

`progress=True` prints the built-in readable lifecycle. `on_event` receives immutable Events for
custom UI, telemetry, finish-reason/refusal inspection, or logging. `limit` selects a bounded
prefix only when the Benchmark permits it. The run below remains disabled by default.

In [ ]:
RUN_EVALUATION = False
events = []

report = (
    client.evaluate(
        [writer, panel],
        benchmark="draco",
        limit=1,
        on_event=events.append,
        progress=True,
    )
    if RUN_EVALUATION
    else None
)
[event.kind for event in events]

## 7. Read the Report as values or a portable artifact

`Report.ok` means the Evaluation produced scored Candidate results without recorded failures.
Results retain the compiled URL4 graph, model operations, aggregate and per-Case usage, finish
reasons, Benchmark grades, Checks, accepted or rejected raw Evidence, failures, and timing.


In [ ]:
if report is not None:
    result = report.candidates["writer"]
    case = result.cases[0]
    report_view = {
        "ok": report.ok,
        "benchmark": report.benchmark,
        "candidate_names": [item.name for item in report.candidates],
        "score": result.score,
        "metrics": dict(result.metrics),
        "url4": result.url4,
        "operations": result.operations,
        "finish_reason": case.finish_reason,
        "grade": case.grade,
        "checks": () if case.grade is None else case.grade.checks,
        "evidence": ()
        if case.grade is None or not case.grade.checks
        else case.grade.checks[0].evidence,
        "failures": report.failures,
        "usage": report.usage,
        "duration_ms": report.duration_ms,
    }
else:
    report_view = "Evaluation disabled — no result to inspect."
report_view

In [ ]:
report.to_json() if report is not None else None

## 8. Handle the public error family

Catch `sf.ScreamingFaceError` for Engine, authentication, planning, connection, and execution
failures. More specific subclasses remain available when recovery differs:

```python
try:
    report = client.evaluate(writer, benchmark="draco", limit=1)
except sf.ProviderConnectionError:
    client.connect()
except sf.PlanningError as exc:
    print(f"Fix the Candidate or Benchmark selection: {exc}")
except sf.ExecutionError as exc:
    print(f"The launched run failed: {exc}")
except sf.ScreamingFaceError as exc:
    print(f"ScreamingFace could not complete the request: {exc}")
```

## 9. Use the asynchronous Client

The asynchronous API mirrors discovery, connections, authentication, and evaluation. Top-level
`await` works in Jupyter, so this metadata-only example is safe to run.

In [ ]:
async with sf.AsyncClient(engine_url="http://127.0.0.1:9108") as async_client:
    async_models = await async_client.models.list()
    async_draco = await async_client.benchmarks.get("draco")

{"model_count": len(async_models), "benchmark": async_draco.id}

## 10. Close the explicit Client

In [ ]:
client.close()
client.closed